In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage ,BaseMessage
from langgraph.checkpoint.memory import MemorySaver
import operator

In [ ]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]


In [ ]:
llm = ChatOpenAI()
def chat_node(state:ChatState):
    #take  user queryt from state
    message = state['message']
    #send to llm
    response = llm.invoke(message)
    #response store to state
    return {"message":[response]}


In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
#add nodes
graph.add_node("chat_node",chat_node)

graph.add_edge(START ,'chat_node')
graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpointer=checkpointer)


In [ ]:
intial_state = {
    'message':[HumanMessage(content="What is stealth Address om monero")]
}


In [ ]:
thread_id = "1"

while True:
    user_message = input("Type here: ")

    print("User:", user_message)

    if user_message.strip().lower() in ["exit", "quit", "bye"]:
        break

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = chatbot.invoke(
        {
            "message": [
                HumanMessage(content=user_message)
            ]
        },
        config=config
    )

    print("AI:", response["message"][-1].content)